# Overview

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

input_path_str: str = os.getenv("FEEDBACK_SOURCE_DIR_PATH") or '.'


In [ ]:
# from nltk.sentiment import SentimentIntensityAnalyzer
# import nltk 

# # nltk.download('vader_lexicon')
# sia = SentimentIntensityAnalyzer()

# def nullable_polarity_scores(text):
#     if pd.isna(text):
#         return {"neg": np.nan, "neu": np.nan, "pos": np.nan, "compound": np.nan}
#     return sia.polarity_scores(str(text))

In [ ]:
from enum import StrEnum, auto
from pathlib import Path
import pandas as pd

class DataFileType(StrEnum):
    CLEAN = auto()
    ENHANCED = auto()

class DataFilePeriod(StrEnum):
    FIRST_MONTH = '20d'
    HALF_YEAR = '6m'
    FULL_YEAR = '1y'

class InputManager:
    _type: DataFileType
    _period: DataFilePeriod
    _SOURCE_DIR_PATH: Path = Path(input_path_str)
    _SUFFIX: str = '.csv'

    def _create_input_path(self, file_type: DataFileType, data_period: DataFilePeriod) -> Path:
        return (self._SOURCE_DIR_PATH / (file_type + data_period).upper()).with_suffix(self._SUFFIX)

    def get_input_data(self, file_type: DataFileType, data_period: DataFilePeriod) -> pd.DataFrame:
        path: Path = self._create_input_path(file_type, data_period)
        df = pd.read_csv(path, index_col=0)
        return df

inputs: InputManager = InputManager()
preferred_file_type: DataFileType = DataFileType.ENHANCED

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class PreprocessingStrategy:
    type_map: Dict[str, Any] = field(default_factory=dict)  # simple dtypes
    datetime_cols: List[str] = field(default_factory=list)
    categorical_cols: List[str] = field(default_factory=list)
    ordinal_cols: Dict[str, List[Any]] = field(default_factory=dict)    # Map column name to an ordered list of categories, lowest to highest

@dataclass
class DataPreprocessor:
    _df: pd.DataFrame

    def set_dtypes(self, strategy: PreprocessingStrategy) -> None:
        df: pd.DataFrame = self._df.copy()

        # combined_datetime_col: pd.Series = df['posting_day'] + ' ' + df['posting_hour']
        # combined_datetime_col = pd.to_datetime(combined_datetime_col, format="%d %b %Y %H:%M").rename("posting_timestamp")
        # df['combined_datetime'] = combined_datetime_col
        for col in strategy.datetime_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col])

        for col in strategy.categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype("category")

        for col, categories in strategy.ordinal_cols.items():
            if col in df.columns:
                # Enforce exact order and flag as ordered
                ordinal_type = pd.CategoricalDtype(categories=categories, ordered=True)
                df[col] = df[col].astype(ordinal_type)

        valid_type_map = {col: dtype for col, dtype in strategy.type_map.items() if col in df.columns}
        df = df.astype(valid_type_map)

        self._df = df



In [ ]:
current_data_period: DataFilePeriod = DataFilePeriod.FIRST_MONTH

df = inputs.get_input_data(preferred_file_type, current_data_period)
df.insert(0, 'posting_timestamp', df['posting_day'] + ' ' + df['posting_hour'])
df.drop(['posting_day', 'posting_hour'], axis=1, inplace=True)

In [ ]:
strategy: PreprocessingStrategy = PreprocessingStrategy(
    type_map={
        'positive_feedback': 'string[pyarrow]', 
        'negative_feedback': 'string[pyarrow]', 
        'positive_feedback_EN': 'string[pyarrow]', 
        'negative_feedback_EN': 'string[pyarrow]',
        'positive_feedback_EN_neg': 'float32',
        'positive_feedback_EN_neu': 'float32',
        'positive_feedback_EN_pos': 'float32',
        'positive_feedback_EN_compound': 'float32',
        'negative_feedback_EN_neg': 'float32',
        'negative_feedback_EN_neu': 'float32',
        'negative_feedback_EN_pos': 'float32',
        'negative_feedback_EN_compound': 'float32',
    },
    datetime_cols=['posting_timestamp'],
    categorical_cols=['survey_type', 'department'],
    ordinal_cols={'onboarding_rating': list(range(1,6))}
)

preprocessor: DataPreprocessor = DataPreprocessor(df)
preprocessor.set_dtypes(strategy)
df = preprocessor._df
df

In [ ]:
print(df.shape)
print('---\n')
print(df.info())
print('---\n')
print(df.head())
print('---\n')
print('The indicies are unique: ', df.index.nunique() == len(df))

Datetime variables require specialized analysis because time functions both as an index (ordering events sequentially) and a feature (reflecting cyclical human behavior and seasonality).

The main goal of datetime univariate analysis is to evaluate data continuity, spot missing time gaps, assess time resolution, and understand periodic patterns.

In [ ]:
ts = df['posting_timestamp']
t_min, t_max = ts.min(), ts.max()
total_span = (t_max - t_min)
time_deltas = ts.diff()

def td_format(td_object):
    seconds = int(td_object.total_seconds())
    periods = [
        ('year',        60*60*24*365),
        ('month',       60*60*24*30),
        ('day',         60*60*24),
        ('hour',        60*60),
        ('minute',      60),
        ('second',      1)
    ]

    strings=[]
    for period_name, period_seconds in periods:
        if seconds > period_seconds:
            period_value , seconds = divmod(seconds, period_seconds)
            has_s = 's' if period_value > 1 else ''
            strings.append("%s %s%s" % (period_value, period_name, has_s))

    return " ".join(strings)

print(f'''
Temporal coverage
    t_min: {t_min:%Y/%m/%d}
    t_max: {t_max:%Y/%m/%d}
    total_span: {td_format(total_span)}
---
Sampling Interval & Resolution
    median difference between answer times: {td_format(time_deltas.median())}
    average difference between answer times: {td_format(time_deltas.mean())}
''')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = df.dropna(subset=['posting_timestamp']).copy()

df_clean['Year-Month'] = [ts.strftime('%Y-%m') for ts in df_clean['posting_timestamp']]
df_clean['DayOfWeek']  = [ts.strftime('%a') for ts in df_clean['posting_timestamp']]
df_clean['Hour']       = [ts.hour for ts in df_clean['posting_timestamp']]

day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

monthly_map = {}
for ym in df_clean['Year-Month']:
    monthly_map[ym] = monthly_map.get(ym, 0) + 1

if monthly_map:
    min_ym = min(monthly_map.keys())
    max_ym = max(monthly_map.keys())
    
    start_y, start_m = map(int, min_ym.split('-'))
    end_y, end_m     = map(int, max_ym.split('-'))
    
    all_months = []
    curr_y, curr_m = start_y, start_m
    while (curr_y, curr_m) <= (end_y, end_m):
        all_months.append(f"{curr_y:04d}-{curr_m:02d}")
        curr_m += 1
        if curr_m > 12:
            curr_m = 1
            curr_y += 1
            
    monthly_keys = all_months
    monthly_vals = [monthly_map.get(ym, 0) for ym in monthly_keys]
else:
    monthly_keys = []
    monthly_vals = []

daily_map = {}
for d in df_clean['DayOfWeek']:
    daily_map[d] = daily_map.get(d, 0) + 1

daily_vals = [daily_map.get(d, 0) for d in day_order]

hourly_map = {}
for h in df_clean['Hour']:
    hourly_map[h] = hourly_map.get(h, 0) + 1

hourly_vals = [hourly_map.get(h, 0) for h in range(24)]

y_max = max(max(monthly_vals, default=0), max(daily_vals, default=0), max(hourly_vals, default=0)) * 1.15

fig, axes = plt.subplots(
    1, 3, 
    figsize=(18, 5), 
    sharey=True, 
    gridspec_kw={'width_ratios': [1.8, 0.6, 1.6]}
)

sns.barplot(
    x=monthly_keys,
    y=monthly_vals,
    hue=monthly_keys,
    legend=False,
    ax=axes[0],
    palette='Blues_d'
)
axes[0].set_title('Post Frequency by Month')
axes[0].set_ylabel('Number of Posts')
axes[0].set_xlabel('Year-Month')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    x=day_order, 
    y=daily_vals, 
    hue=day_order, 
    legend=False, 
    ax=axes[1], 
    palette='Blues_d'
)
axes[1].set_title('Post Frequency by Day')
axes[1].set_xlabel('DayOfWeek')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(
    x=list(range(24)), 
    y=hourly_vals, 
    hue=list(range(24)), 
    legend=False, 
    ax=axes[2], 
    palette='viridis'
)
axes[2].set_title('Post Frequency by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_xticks([i - 0.5 for i in range(25)])
axes[2].set_xticklabels([f"{h:02d}:00" for h in range(25)], rotation=45)

for ax in axes:
    for container in ax.containers:
        labels = [f'{int(v)}' if v > 0 else '' for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=2, fontsize=8)

axes[0].set_ylim(0, y_max)

plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.dropna(subset=['posting_timestamp']).copy()

df_clean['DayOfWeek'] = df_clean['posting_timestamp'].dt.day_name()
labels = ['00:00-03:59', '04:00-07:59', '08:00-11:59', '12:00-15:59', '16:00-19:59', '20:00-23:59']

df_clean['Hour_4H'] = pd.cut(
    df_clean['posting_timestamp'].dt.hour, 
    bins=[-1, 3, 7, 11, 15, 19, 23], 
    labels=labels
)

day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

heatmap_data = pd.crosstab(
    df_clean['DayOfWeek'], 
    df_clean['Hour_4H']
).reindex(day_order)

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True, fmt='d', cbar_kws={'label': 'Post Count'})
plt.title('Post Frequency Heatmap (Day of Week vs. Hour of Day)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()